# Preparación de datos (Favorita)

Configuración única por integrante, antes de correr esta sección:

1. Aceptar las reglas de la competencia en Kaggle: https://www.kaggle.com/competitions/store-sales-time-series-forecasting/rules
2. Generar un token en https://www.kaggle.com/settings/api → **Generate New Token**. El token (empieza con `KGAT_`) se muestra una sola vez: cópialo.
3. Ejecutar las celdas de abajo. La celda de credenciales va a pedir que pegues ese token (queda solo en el runtime remoto de Colab, nunca en el repo).

El CLI de `kaggle` ya no usa `kaggle.json` con `username`+`key` (Basic Auth); usa este token único vía `~/.kaggle/access_token`. `files.upload()` tampoco funciona desde la extensión de Colab en VS Code (depende de la interfaz web de colab.research.google.com), por eso el token se pide con `getpass`.

In [18]:
!pip install -q -U kaggle

In [19]:
import os
from getpass import getpass

kaggle_token = getpass("Kaggle API token (KGAT_...): ")

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
token_path = os.path.join(kaggle_dir, "access_token")
with open(token_path, "w") as f:
    f.write(kaggle_token.strip())
os.chmod(token_path, 0o600)

In [20]:
import os

os.makedirs("/content/data", exist_ok=True)
!kaggle competitions download -c store-sales-time-series-forecasting -p /content/data
!unzip -o -q /content/data/store-sales-time-series-forecasting.zip -d /content/data

100% 21.4M/21.4M [00:00<00:00, 117MB/s] 



In [21]:
import pandas as pd

train = pd.read_csv("/content/data/train.csv", parse_dates=["date"])
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


## Selección de tienda y familias

Criterio (issue #2): elegir la tienda de mayor volumen de ventas entre las que tienen menos de 20% de días en cero (umbral laxo a nivel tienda, porque mezclar las 33 familias —incluidas nichos casi siempre en cero, como libros o cuidado de bebé— infla el % agregado). Para las familias dentro de esa tienda, umbral más estricto de 10% (ahí cada serie es la que de verdad se modela).

In [22]:
stores = pd.read_csv("/content/data/stores.csv")

store_stats = (
    train.groupby("store_nbr")
    .agg(volumen=("sales", "sum"), pct_ceros=("sales", lambda s: (s == 0).mean()))
    .reset_index()
)
store_stats["pct_ceros"] = (store_stats["pct_ceros"] * 100).round(2)
store_stats = store_stats.merge(stores, on="store_nbr").sort_values("volumen", ascending=False)
store_stats

,store_nbr,volumen,pct_ceros,city,state,type,cluster
43,44,6.208755e+07,17.95,Quito,Pichincha,A,5
44,45,5.449801e+07,19.36,Quito,Pichincha,A,11
46,47,5.094831e+07,18.95,Quito,Pichincha,A,14
2,3,5.048191e+07,18.14,Quito,Pichincha,D,8
48,49,4.342010e+07,19.35,Quito,Pichincha,A,11
45,46,4.189606e+07,19.32,Quito,Pichincha,A,14
47,48,3.593313e+07,21.32,Quito,Pichincha,A,14
50,51,3.291149e+07,20.12,Guayaquil,Guayas,A,17
7,8,3.049429e+07,19.93,Quito,Pichincha,D,8
49,50,2.865302e+07,20.65,Ambato,Tungurahua,A,14


In [ ]:
STORE_ZERO_THRESHOLD = 20  # % de días en cero, a nivel tienda (mezcla las 33 familias)

candidatos = store_stats[store_stats["pct_ceros"] < STORE_ZERO_THRESHOLD]
selected_store = candidatos.sort_values("volumen", ascending=False).iloc[0]
selected_store_nbr = int(selected_store["store_nbr"])

print(
    f"Tienda elegida: {selected_store_nbr} "
    f"(volumen={selected_store['volumen']:.0f}, % ceros={selected_store['pct_ceros']:.2f}%, "
    f"ciudad={selected_store['city']}, tipo={selected_store['type']}, cluster={selected_store['cluster']})"
)

In [26]:
family_stats = (
    train[train["store_nbr"] == selected_store_nbr]
    .groupby("family")
    .agg(volumen=("sales", "sum", ), pct_ceros=("sales", lambda s: (s == 0).mean()))
    .reset_index()
)
family_stats["pct_ceros"] = (family_stats["pct_ceros"] * 100).round(2)
family_stats["volumen"] = family_stats["volumen"].round(2)
family_stats.sort_values("volumen", ascending=False)

,family,volumen,pct_ceros
12,GROCERY I,16386055.04,0.30
3,BEVERAGES,13417859.00,0.30
30,PRODUCE,10666058.48,4.81
7,CLEANING,4308260.00,0.30
8,DAIRY,4277954.00,0.30
5,BREAD/BAKERY,2333058.28,0.30
28,POULTRY,2086943.63,0.30
24,MEATS,1636611.01,0.30
10,EGGS,1091864.00,0.30
11,FROZEN FOODS,1021094.49,0.30


In [27]:
FAMILY_ZERO_THRESHOLD = 10  # % de días en cero, a nivel familia (la serie que se modela)
N_FAMILIES = 5

selected_families = (
    family_stats[family_stats["pct_ceros"] < FAMILY_ZERO_THRESHOLD]
    .sort_values("volumen", ascending=False)
    .head(N_FAMILIES)
)

print(f"Familias elegidas para la tienda {selected_store_nbr}:")
selected_families

Familias elegidas para la tienda 44:


,family,volumen,pct_ceros
12,GROCERY I,16386055.04,0.30
3,BEVERAGES,13417859.00,0.30
30,PRODUCE,10666058.48,4.81
7,CLEANING,4308260.00,0.30
8,DAIRY,4277954.00,0.30
